In [1]:
# IMPORT FILES FROM DATASET
import os
from json.decoder import JSONArray

from jupyter_server.utils import fetch

files = [file for file in os.listdir('data/') if file.endswith(".pgn")]

In [2]:
len(files)

1

In [3]:
from chess import pgn

def load_pgn(file):
    games = []
    with open(file, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)

    return games

In [4]:
from tqdm import tqdm

games = []

for file in tqdm(files):
    games.extend(load_pgn('data/' + file))

100%|██████████| 1/1 [00:17<00:00, 17.73s/it]


In [5]:
print(type(games[0]))

<class 'chess.pgn.Game'>


In [6]:
len(games)

8728

In [14]:
# IMPORTS FOR NEURAL NETWORK
import numpy as np
from chess import Board
from chess import PAWN, KNIGHT, BISHOP, ROOK, KING, QUEEN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Input, BatchNormalization, MaxPooling2D, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.regularizers import l2

In [29]:
# turn the board into a matrix
def board_to_matrix(board : Board):
    matrix = np.zeros((8, 8, 15))
    piece_map = board.piece_map()

    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        piece_eval = 1 if piece.color else -1

        if piece.piece_type == PAWN:
            piece_eval *= 10
        elif piece.piece_type == KNIGHT:
            piece_eval *= 30
        elif piece.piece_type == BISHOP:
            piece_eval *= 30
        elif piece.piece_type == ROOK:
            piece_eval *= 50
        elif piece.piece_type == QUEEN:
            piece_eval *= 90
        elif piece.piece_type == KING:
            piece_eval *= 900

        matrix[row, col, piece_type + piece_color] = piece_eval

    legal_moves = board.legal_moves
    pseudo_moves = board.pseudo_legal_moves

    for move in legal_moves:
        to_square = move.to_square
        row_to, col_to = divmod(to_square, 8)
        matrix[row_to, col_to, 12] = 1

        if board.piece_at(move.to_square):
            matrix[row_to, col_to, 13] = 1

    for move in pseudo_moves:
        to_square = move.to_square
        row_to, col_to = divmod(to_square, 8)
        matrix[row_to, col_to, 14] = 1

    return matrix

# inputs for possible moves
def input_for_nn(games):
    X = []
    y = []
    for game in games:
        print(f'game {games.index(game) + 1} / {len(games)}')
        board = game.board()
        for move in game.mainline_moves():
            X.append(board_to_matrix(board))
            y.append(move.uci())
            board.push(move)
    return X, y

# convert moves
def encode_moves(moves):
    print("encoding...")
    move_to_int = {move: idx for idx, move in enumerate(set(moves))}
    return [move_to_int[move] for move in moves], move_to_int


In [30]:
X, y = input_for_nn(games[:2000])
y, move_to_int = encode_moves(y)
print(f'length of move_to_int:{len(move_to_int)}')
y = to_categorical(y, num_classes=len(move_to_int))
print(f'length of y:{len(y)}')

game 1 / 2000
game 2 / 2000
game 3 / 2000
game 4 / 2000
game 5 / 2000
game 6 / 2000
game 7 / 2000
game 8 / 2000
game 9 / 2000
game 10 / 2000
game 11 / 2000
game 12 / 2000
game 13 / 2000
game 14 / 2000
game 15 / 2000
game 16 / 2000
game 17 / 2000
game 18 / 2000
game 19 / 2000
game 20 / 2000
game 21 / 2000
game 22 / 2000
game 23 / 2000
game 24 / 2000
game 25 / 2000
game 26 / 2000
game 27 / 2000
game 28 / 2000
game 29 / 2000
game 30 / 2000
game 31 / 2000
game 32 / 2000
game 33 / 2000
game 34 / 2000
game 35 / 2000
game 36 / 2000
game 37 / 2000
game 38 / 2000
game 39 / 2000
game 40 / 2000
game 41 / 2000
game 42 / 2000
game 43 / 2000
game 44 / 2000
game 45 / 2000
game 46 / 2000
game 47 / 2000
game 48 / 2000
game 49 / 2000
game 50 / 2000
game 51 / 2000
game 52 / 2000
game 53 / 2000
game 54 / 2000
game 55 / 2000
game 56 / 2000
game 57 / 2000
game 58 / 2000
game 59 / 2000
game 60 / 2000
game 61 / 2000
game 62 / 2000
game 63 / 2000
game 64 / 2000
game 65 / 2000
game 66 / 2000
game 67 / 2000
game

In [31]:
print(f'length of X:{len(X)}')
X = np.array(X)
print(f'length of X:{len(X)}')

length of X:188808
length of X:188808


In [14]:
from tensorflow.keras.models import load_model
model = load_model('hikarubot_23.keras')

I0000 00:00:1745908785.580663    3340 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2274 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [36]:

model = Sequential([
    Input(shape=(8, 8,15)),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Flatten(),
    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(len(move_to_int), activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_10 (Conv2D)              │ (None, 8, 8, 64)       │         8,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 8, 8, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1826)           │       469,282 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,650,018 (10.11 MB)

 Trainable params: 2,649,634 (10.11 MB)

 Non-trainable params: 384 (1.50 KB)

In [37]:
history = model.fit(X, y, epochs=50, validation_split=0.1, batch_size=64)
model.save('hikarubot_2.keras')

Epoch 1/50


2025-04-30 16:25:09.906539: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1135', 616 bytes spill stores, 620 bytes spill loads

2025-04-30 16:25:09.934666: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1135', 272 bytes spill stores, 272 bytes spill loads

2025-04-30 16:25:10.021791: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1135', 56 bytes spill stores, 56 bytes spill loads

2025-04-30 16:25:10.273686: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1135', 424 bytes spill stores, 424 bytes spill loads

2025-04-30 16:25:10.357823: I exte

2654/2656 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0131 - loss: 7.1087

2025-04-30 16:25:24.285859: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1135', 120 bytes spill stores, 120 bytes spill loads

2025-04-30 16:25:24.452342: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1135', 260 bytes spill stores, 260 bytes spill loads



2656/2656 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0131 - loss: 7.1085

2025-04-30 16:25:27.075708: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_122', 104 bytes spill stores, 104 bytes spill loads

2025-04-30 16:25:27.174886: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_122', 12 bytes spill stores, 12 bytes spill loads

2025-04-30 16:25:27.602348: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_122', 88 bytes spill stores, 88 bytes spill loads



2656/2656 ━━━━━━━━━━━━━━━━━━━━ 22s 6ms/step - accuracy: 0.0131 - loss: 7.1084 - val_accuracy: 0.0415 - val_loss: 6.2818
Epoch 2/50
2656/2656 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0342 - loss: 6.3312 - val_accuracy: 0.0712 - val_loss: 5.8484
Epoch 3/50
2656/2656 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0567 - loss: 5.9714 - val_accuracy: 0.0873 - val_loss: 5.5561
Epoch 4/50
2656/2656 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0779 - loss: 5.7021 - val_accuracy: 0.1086 - val_loss: 5.3016
Epoch 5/50
2656/2656 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0932 - loss: 5.4689 - val_accuracy: 0.1347 - val_loss: 5.0244
Epoch 6/50
2656/2656 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.1094 - loss: 5.2767 - val_accuracy: 0.1473 - val_loss: 4.8313
Epoch 7/50
2656/2656 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.1214 - loss: 5.1024 - val_accuracy: 0.1569 - val_loss: 4.6802
Epoch 8/50
2656/2656 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.1317 - loss: 4.9734 - val

In [38]:
# Load model
from tensorflow.keras.models import load_model
model = load_model('./hikarubot_2.keras')

In [39]:
int_to_move = dict(zip(move_to_int.values(), move_to_int.keys()))

def predict_move(board : Board):
    board_matrix = board_to_matrix(board).reshape(1, 8, 8, 15)
    prediction = model.predict(board_matrix)
    move = int_to_move[np.argmax(prediction)]
    return move

In [40]:

board = Board()


In [75]:
# while not board.is_game_over():
#     board.push_uci(input("Your move:"))

next_move = predict_move(board)
board.push_uci(next_move)

print(f"Predicted move: {next_move}")
print(board)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


IllegalMoveError: illegal uci: 'b8d7' in r1bq1rk1/ppp3pp/1n2pb2/8/8/2N2B2/PP1N1PPP/R2QR1K1 b - - 1 16

In [67]:
array = np.dict(move_to_int)
np.save('move_to_int.npy', array)

In [68]:
print(move_to_int)

{'b2c3': 0, 'd7a4': 1, 'e2f4': 2, 'c1e3': 3, 'c4a2': 4, 'd4g7': 5, 'g7e8': 6, 'd5f4': 7, 'g5g4': 8, 'b1a2': 9, 'e1f2': 10, 'b3a1': 11, 'g2g5': 12, 'f3b3': 13, 'c7f7': 14, 'h4h8': 15, 'f6e6': 16, 'f4h6': 17, 'f4d2': 18, 'a3b3': 19, 'c2d3': 20, 'f2a2': 21, 'a5c5': 22, 'c7c8n': 23, 'a6b5': 24, 'd6e8': 25, 'c8e6': 26, 'h6g5': 27, 'g2h3': 28, 'f2h2': 29, 'a8a6': 30, 'e3c3': 31, 'e3g4': 32, 'f2b6': 33, 'c6e5': 34, 'g7f6': 35, 'b1c1': 36, 'b1d2': 37, 'f3a3': 38, 'h6a6': 39, 'b8c8': 40, 'h4d8': 41, 'b2f2': 42, 'c2h7': 43, 'g5h5': 44, 'f3f7': 45, 'e6f8': 46, 'b6b3': 47, 'e1g1': 48, 'a6f1': 49, 'f3a8': 50, 'b6e6': 51, 'h1h8': 52, 'f7a7': 53, 'h3c8': 54, 'f6f2': 55, 'e3g2': 56, 'f4e3': 57, 'b8g3': 58, 'g7e5': 59, 'e7d8': 60, 'd8d1': 61, 'h6e3': 62, 'g4h4': 63, 'c6c2': 64, 'b6d6': 65, 'd8b7': 66, 'a3b4': 67, 'b7d6': 68, 'f7f1': 69, 'g8f7': 70, 'c4d3': 71, 'd4d1': 72, 'd7d8n': 73, 'f6d7': 74, 'c3d1': 75, 'h5f7': 76, 'f4f5': 77, 'e5a1': 78, 'g7g2': 79, 'b6b7': 80, 'd2h2': 81, 'b2b3': 82, 'e5h8': 83,